ML

In [ ]:
# Load ML-ready dataset

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.feature_selection import (
    mutual_info_classif
)

from sklearn.ensemble import (
    RandomForestClassifier
)

# Load dataset

df = pd.read_csv(
    "titanic_ml_ready.csv"
)

print("Dataset shape:")
print(df.shape)

display(df.head())

Dataset shape:
(1309, 14)


,pclass,survived,sex,age,sibsp,parch,fare,embarked,FamilySize,IsAlone,Has_Cabin,Title,TicketGroupSize,Fare_log
0,1,1,female,29.0000,0,0,211.3375,S,1,1,1,Miss,4,5.358177
1,1,1,male,0.9167,1,2,151.5500,S,4,0,1,Master,6,5.027492
2,1,0,female,2.0000,1,2,151.5500,S,4,0,1,Miss,6,5.027492
3,1,0,male,30.0000,1,2,151.5500,S,4,0,1,Mr,6,5.027492
4,1,0,female,25.0000,1,2,151.5500,S,4,0,1,Mrs,6,5.027492


In [6]:
# Separate features and target

target = "survived"

X = df.drop(
    columns=target
)

y = df[target]

print(
    "X shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

X shape: (1309, 13)
y shape: (1309,)


In [7]:
# Identify feature types

numeric_features = X.select_dtypes(include=np.number).columns.tolist()

categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric:")

print(numeric_features)

print("\nCategorical:")

print(categorical_features)

Numeric:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'FamilySize', 'IsAlone', 'Has_Cabin', 'TicketGroupSize', 'Fare_log']

Categorical:
['sex', 'embarked', 'Title']


In [8]:
# Prepare data for feature selection

# Feature selection will use training data only.



In [9]:
# Stratified train-test split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Train:", X_train.shape)

print("Test:", X_test.shape)

Train: (1047, 13)
Test: (262, 13)


In [10]:
# Verify class balance

print("Full dataset:")

print(y.value_counts(normalize=True))

print("\nTrain:")

print(y_train.value_counts(normalize=True))

print("\nTest:")

print(y_test.value_counts(normalize=True))

Full dataset:
survived
0    0.618029
1    0.381971
Name: proportion, dtype: float64

Train:
survived
0    0.617956
1    0.382044
Name: proportion, dtype: float64

Test:
survived
0    0.618321
1    0.381679
Name: proportion, dtype: float64


In [11]:
# Numeric preprocessing

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [12]:
# Categorical preprocessing

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [13]:
# Combine preprocessing

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [14]:
# Fit preprocessing on train only

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed train:", X_train_processed.shape)

print("Processed test:", X_test_processed.shape)

Processed train: (1047, 20)
Processed test: (262, 20)


In [15]:
# Get transformed feature names

feature_names = preprocessor.get_feature_names_out()

print(feature_names)

['num__pclass' 'num__age' 'num__sibsp' 'num__parch' 'num__fare'
 'num__FamilySize' 'num__IsAlone' 'num__Has_Cabin' 'num__TicketGroupSize'
 'num__Fare_log' 'cat__sex_female' 'cat__sex_male' 'cat__embarked_C'
 'cat__embarked_Q' 'cat__embarked_S' 'cat__Title_Master' 'cat__Title_Miss'
 'cat__Title_Mr' 'cat__Title_Mrs' 'cat__Title_Rare']


In [16]:
# Mutual information scores

mi_scores = mutual_info_classif(X_train_processed, y_train, random_state=42)

mi_results = pd.DataFrame({"feature": feature_names, "mi_score": mi_scores})

mi_results = mi_results.sort_values("mi_score", ascending=False)

display(mi_results)

,feature,mi_score
17,cat__Title_Mr,0.146133
10,cat__sex_female,0.143584
4,num__fare,0.118717
9,num__Fare_log,0.113259
11,cat__sex_male,0.111134
18,cat__Title_Mrs,0.090296
8,num__TicketGroupSize,0.071253
7,num__Has_Cabin,0.052664
5,num__FamilySize,0.035920
16,cat__Title_Miss,0.033282


In [17]:
# Random forest importance

rf_selector = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")

rf_selector.fit(X_train_processed, y_train)

rf_importance = pd.DataFrame({"feature": feature_names, "importance": rf_selector.feature_importances_})

rf_importance = rf_importance.sort_values("importance", ascending=False)

display(rf_importance)

,feature,importance
1,num__age,0.212757
4,num__fare,0.139375
9,num__Fare_log,0.136099
11,cat__sex_male,0.087925
17,cat__Title_Mr,0.080027
10,cat__sex_female,0.073881
8,num__TicketGroupSize,0.041763
0,num__pclass,0.037619
5,num__FamilySize,0.034088
7,num__Has_Cabin,0.027864


In [18]:
# Compare selection scores

feature_selection = mi_results.merge(rf_importance, on="feature")

feature_selection["combined_score"] = feature_selection["mi_score"].rank(ascending=False) + feature_selection["importance"].rank(ascending=False)

feature_selection = feature_selection.sort_values("combined_score")

display(feature_selection)

,feature,mi_score,importance,combined_score
2,num__fare,0.118717,0.139375,5.0
0,cat__Title_Mr,0.146133,0.080027,6.0
3,num__Fare_log,0.113259,0.136099,7.0
1,cat__sex_female,0.143584,0.073881,8.0
4,cat__sex_male,0.111134,0.087925,9.0
6,num__TicketGroupSize,0.071253,0.041763,14.0
15,num__age,0.007192,0.212757,17.0
5,cat__Title_Mrs,0.090296,0.020059,18.0
7,num__Has_Cabin,0.052664,0.027864,18.0
8,num__FamilySize,0.035920,0.034088,18.0


In [19]:
# Review selected features

top_features = feature_selection.head(20)

display(top_features)

,feature,mi_score,importance,combined_score
2,num__fare,0.118717,0.139375,5.0
0,cat__Title_Mr,0.146133,0.080027,6.0
3,num__Fare_log,0.113259,0.136099,7.0
1,cat__sex_female,0.143584,0.073881,8.0
4,cat__sex_male,0.111134,0.087925,9.0
6,num__TicketGroupSize,0.071253,0.041763,14.0
15,num__age,0.007192,0.212757,17.0
5,cat__Title_Mrs,0.090296,0.020059,18.0
7,num__Has_Cabin,0.052664,0.027864,18.0
8,num__FamilySize,0.035920,0.034088,18.0


In [20]:
# Final preprocessing pipeline

final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        )
                    ),
                    (
                        "scaler",
                        StandardScaler()
                    )
                ]
            ),
            numeric_features
        ),

        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore"
                        )
                    )
                ]
            ),
            categorical_features
        )
    ]
)

print(
    final_preprocessor
)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['pclass', 'age', 'sibsp', 'parch', 'fare',
                                  'FamilySize', 'IsAlone', 'Has_Cabin',
                                  'TicketGroupSize', 'Fare_log']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['sex', 'embarked', 'Title'])])
